# hy3d worker (Hunyuan3D-2 on Kaggle)

Serves image-to-3D jobs from the `queue` branch of the GitHub repo. Jobs are submitted with `python3 hy3d.py submit <image>`.

**Before running:**
1. *Settings → Accelerator*: **GPU T4 x2** (P100 is too old for current Kaggle PyTorch builds).
2. *Settings → Internet*: **on**.
3. *Add-ons → Secrets*: add `GITHUB_TOKEN`, a fine-grained GitHub token with **Contents: read and write** on the repo, and tick it for this notebook.

Then **Run All**. The last cell runs until you stop it. To keep the worker going without the browser open, use **Save Version → Save & Run All (Commit)**. That runs headless for up to 12 h and uses your weekly GPU quota while it is up.

In [ ]:
REPO = "Eslezer/3DENGINE"   # owner/name of the repo holding the queue branch
TEXTURE = False              # True: also build the texture-painting CUDA extensions (+~10 min setup)

import base64, os, subprocess
from kaggle_secrets import UserSecretsClient

os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
auth = base64.b64encode(f"x-access-token:{os.environ['GITHUB_TOKEN']}".encode()).decode()
hdr = ["-c", f"http.extraheader=AUTHORIZATION: basic {auth}"]
SRC = "/tmp/hy3d/3DENGINE"   # outside /kaggle/working so nothing here lands in notebook output
if os.path.exists(SRC):
    subprocess.run(["git", *hdr, "-C", SRC, "pull", "-q"], check=True)
else:
    subprocess.run(["git", *hdr, "clone", "-q", "--depth", "1", "-b", "main",
                    f"https://github.com/{REPO}.git", SRC], check=True)
os.chdir(SRC)
print("ok:", os.listdir(SRC))

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python worker.py --setup-only {"--texture" if TEXTURE else ""}

In [ ]:
# Serves jobs forever. Model weights download from Hugging Face on the first job of each preset.
!python -u worker.py --repo {REPO}